# Bloc 2.6 — Orchestration and pipeline monitoring

**Decision problem:** how do we know the pipeline is fresh, complete, and safe to use?

Output: monitoring report.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]: d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"
def load_clean_long():
    p=OUT/"bloc1"/"clean_trends_long.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.melt(id_vars="date", var_name="signal", value_name="interest")
def load_clean_wide():
    p=OUT/"bloc1"/"clean_trends_wide.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.sort_values("date")

In [ ]:
artifacts=[OUT/"bloc1"/"clean_trends_long.csv",OUT/"bloc1"/"model_metrics.csv",OUT/"bloc2"/"format_comparison.csv",OUT/"bloc2"/"partition_summary.csv"]
rows=[]
for p in artifacts:
    rows.append({"artifact":str(p.relative_to(ROOT)),"exists":p.exists(),"bytes":p.stat().st_size if p.exists() else 0})
monitor=pd.DataFrame(rows)
monitor["status"]=np.where(monitor["exists"] & (monitor["bytes"]>0),"ok","missing")
monitor.to_csv(OUT/"bloc2"/"pipeline_monitoring_report.csv", index=False)
monitor

## Exercise

Add one freshness check and one schema check.

## Conclusion

Monitoring converts pipelines from scripts into reliable decision infrastructure.